In [ ]:
# Locate the project when Jupyter starts in the root or notebooks directory.
from pathlib import Path
import os
import sys

project_root = next(
    folder for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / "project_paths.py").is_file()
)
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from project_paths import DATA_DIR, MODELS_DIR, OUTPUT_DIR

data_file_path = DATA_DIR / "sms_spam_collection" / "SMSSpamCollection.tsv"


In [1]:
import pandas as pd

df = pd.read_csv(
    data_file_path, sep="\t",header=None,names=["Label","text"]
)
df

NameError: name 'data_file_path' is not defined

In [2]:
import torch
from torch.utils.data import Dataset
import pandas as pd
import tiktoken

class SpamDataset(Dataset):
    def __init__(self, csv_file, tokenizer, max_length=None,
                 pad_token_id=50256):
        self.data = pd.read_csv(csv_file)

        self.encoded_texts = [
            tokenizer.encode(text) for text in self.data["text"]
        ]

        if max_length is None:
            self.max_length = self._longest_encoded_length()
        else:
            self.max_length = max_length

            self.encoded_texts = [
                encoded_text[:self.max_length]
                for encoded_text in self.encoded_texts
            ]

        self.encoded_texts = [
            encoded_text + [pad_token_id] *
            (self.max_length - len(encoded_text))
            for encoded_text in self.encoded_texts
        ]

    def __getitem__(self, index):
        encoded = self.encoded_texts[index]
        label = self.data.iloc[index]["Label"]
        return (
            torch.tensor(encoded, dtype=torch.long),
            torch.tensor(label, dtype=torch.long)
        )

    def __len__(self):
        return len(self.data)

    def _longest_encoded_length(self):
        max_length = 0
        for encoded_text in self.encoded_texts:
            encoded_length = len(encoded_text)
            if encoded_length > max_length:
                max_length = encoded_length
        return max_length


In [4]:
tokenizer = tiktoken.get_encoding("gpt2")

In [5]:
# 1. 验证自动计算最大长度的逻辑
train_dataset = SpamDataset(csv_file=DATA_DIR / "sms_spam_collection" / "train.csv", max_length=None, tokenizer=tokenizer)
print(f"训练集自动计算的最大长度: {train_dataset.max_length}")

# 2. 对比一下，如果我强制指定长度呢？
train_dataset_fixed = SpamDataset(csv_file=DATA_DIR / "sms_spam_collection" / "train.csv", max_length=4, tokenizer=tokenizer)
print(f"强制指定后的最大长度: {train_dataset_fixed.max_length}")

训练集自动计算的最大长度: 120
强制指定后的最大长度: 4
